# 23 Train Audio Heads

In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix

from woofdecoder.config import REPO_ROOT
from wooftrainer.trainers.audio_heads import save_audio_heads, train_audio_heads

## Train Preparation

In [2]:
ENCODER = "beats_iter3_plus_as2m"
LAYER = "mean" # notebook 20
HEADS_DIR = REPO_ROOT/"models"/"audio"/ENCODER

## Training

In [4]:
heads = train_audio_heads(ENCODER, LAYER)
print(heads.summary.round(5).to_string(index=False))
print("\nnudge used:", heads.summary.loc[2, "l2"] != "none")

     head      C   l2  temperature  before_log_loss  before_macro_f1  after_log_loss  after_macro_f1
  arousal 0.0010           0.85746          0.89024          0.57157         0.88714         0.57157
  valence 0.0003           0.83079          1.03770          0.46126         1.03605         0.46126
situation 0.0003 none      0.78589          1.15417          0.51874         1.14319         0.51874

nudge used: False


## Per Class and Confusion Matrix

In [5]:
classes = heads.situation.classes
rows = heads.situation_rows.reset_index(drop=True)
true = rows["mapped_category"].to_numpy()
predicted = np.array(classes)[heads.situation_oof_probs.argmax(axis=1)]

print(classification_report(true, predicted, labels=classes, digits=5, zero_division=0))

counts = pd.DataFrame(confusion_matrix(true, predicted, labels=classes), index=classes, columns=classes)
row_share = counts.div(counts.sum(axis=1), axis=0)
print("rows = true category, columns = predicted")
print(row_share.round(2).to_string())

                   precision    recall  f1-score   support

 alert_aggressive    0.31806   0.46053   0.37626      1824
attention_seeking    0.34495   0.36270   0.35360      2429
distressed_lonely    0.11975   0.91935   0.21190        62
            other    0.42510   0.45486   0.43948      3157
  playful_excited    0.62059   0.44488   0.51825      5570

         accuracy                        0.43644     13042
        macro avg    0.36569   0.52847   0.37990     13042
     weighted avg    0.47724   0.43644   0.44720     13042

rows = true category, columns = predicted
                   alert_aggressive  attention_seeking  distressed_lonely  other  playful_excited
alert_aggressive               0.46               0.20               0.04   0.16             0.14
attention_seeking              0.21               0.36               0.03   0.16             0.24
distressed_lonely              0.02               0.02               0.92   0.05             0.00
other                          0

Recall is only 0.36–0.46. About one bark in five is mistaken for each neighbouring category. (attention_seeking is the hardest: 24% become playful, 21% alert)

distressed_lonely is found too well (possible scenarios)
1. Distress really sounds distinct: whining and howling differ acoustically from barking. This is supported by the fact that Act_Env other( rest, shower, where dogs often whine) gets the most false distressed calls (4.5%).
2. A source shortcut: every distressed clip is YouTube, and nothing else in training is. The head could be recognising “YouTube audio” (compression, phone microphones, rooms) instead of distress.

Act_Env CV is split by clip with AI-generated labels: optimistic.

## By Source, per video for self-labeled clips

In [6]:
for source in ("act_env", "self_labeled"):
    keep = (rows["source"] == source).to_numpy()
    print(f"\n{source}: {keep.sum()} rows")
    print(pd.crosstab(true[keep], predicted[keep], rownames=["true"], colnames=["predicted"]))
 
# One answer per self-labeled video: average its windows' probabilities (what the app would see).
mine = rows["source"] == "self_labeled"
video_ids = rows.loc[mine, "clip_id"].str.split("@").str[0]
video_probs = pd.DataFrame(heads.situation_oof_probs[mine.to_numpy()], columns=classes)
video_probs["video"] = video_ids.to_numpy()
per_video = video_probs.groupby("video").mean()
per_video["predicted"] = per_video[classes].idxmax(axis=1)
print("\nper video (distressed_lonely), out-of-fold:")
print(per_video.round(2).to_string())
print("\nvideos predicted distressed_lonely:",
      (per_video["predicted"] == "distressed_lonely").sum(), "of", len(per_video))
 
# Act_Env clips wrongly called distressed: the false alarms for this class.
act_env = (rows["source"] == "act_env").to_numpy()
false_alarms = (predicted[act_env] == "distressed_lonely").mean()
print(f"Act_Env clips predicted distressed_lonely: {false_alarms:.1%}")


act_env: 12980 rows
predicted          alert_aggressive  attention_seeking  distressed_lonely  \
true                                                                        
alert_aggressive                840                360                 70   
attention_seeking               517                881                 61   
other                           503                411                143   
playful_excited                 780                901                145   

predicted          other  playful_excited  
true                                       
alert_aggressive     291              263  
attention_seeking    382              588  
other               1436              664  
playful_excited     1266             2478  

self_labeled: 62 rows
predicted          alert_aggressive  attention_seeking  distressed_lonely  \
true                                                                        
distressed_lonely                 1                  1                 57  

Distressed precision is 0.12. 3.2% of Act_Env (about 420 clips) are falsely called distressed, against 57 true windows. That’s mostly a side effect of 13,000 vs 62 rows.

**Source check: not done yet (2026-11-06).**  
All distressed_lonely training audio is YouTube; no other training audio is.   
The out-of-fold result (25/27 videos, recall 0.92) may partly reflect the source rather than distress.  
To do: run the head on non-distressed YouTube clips (fusion train) once they are prepared.